In [ ]:
import os
from pathlib import Path
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import contextily as ctx
import glob
from PIL import Image
import imageio.v3 as iio
import scienceplots

plt.style.use(['nature'])
#plt.style.use(['ieee','no-latex'])

In [ ]:
# Generate static image for testing
# --- Configuration ---
base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
out_dir = "/export/lv9/projects/dws/results/animation/benthos"
spinup = "spinup_06"
date_str = "201501"

#variable = "Y1c" 
#label = "Epibenthos (Y1)"

#variable = "Y2c" 
#label = "Deposit feeders (Y2)"

variable = "Y3c" 
label = "Suspension feeders (Y3)"

#variable = "Y5c" 
#label = "Endobenthos (Y5)"

# Fixed map window: Dutch Wadden Sea to Ems-Dollard
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# Fixed color scale configuration to prevent map flickering
VMIN = 10.0
VMAX = 100000.0

log_norm = mcolors.LogNorm(vmin=VMIN, vmax=VMAX)

# --- Load Data ---
file_path = os.path.join(base_dir, spinup, f"dws_500m.3d.{date_str}.nc")
ds = xr.open_dataset(file_path)

# Extract variable. If the variable is 3D (e.g., pelagic), we safely isolate the bottom layer.
data_var = ds[variable]
if 'level' in data_var.dims:
    data_var = data_var.isel(level=-1) 

# Select the first time step in the daily output file
data_slice = data_var.isel(time=0)

# Extract 2D coordinate arrays and mathematically fill any spatial gaps (NaNs) 
# usually found in the masked land boundaries of the domain mesh.
lon_da = ds.lonc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')
lat_da = ds.latc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')

lon = lon_da.values
lat = lat_da.values

# --- Plotting ---
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.Mercator())
ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

## Add map features
#ax.add_feature(cfeature.LAND, zorder=2, edgecolor='k', facecolor='lightgray')
#ax.add_feature(cfeature.COASTLINE, zorder=2)

# Add the high-resolution background
ctx.add_basemap(ax, 
                crs=ax.projection.to_string(), 
                #source=ctx.providers.CartoDB.Positron,
                source=ctx.providers.Esri.WorldShadedRelief,
                zorder=1,
               )

txt = ax.texts[-1]
txt.set_position([0.99,0.02])
txt.set_ha('right')

# Plot spatial distribution
mesh = ax.pcolormesh(lon, lat, data_slice.values,
                     transform=ccrs.PlateCarree(),
                     cmap=cmocean.cm.matter,
                     norm=log_norm,
                     shading='gouraud',
                     zorder=2)



# Formatting
cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
cbar.ax.tick_params(labelsize=10)

# Dynamically fetch units if available in the netCDF metadata
#units = ds[variable].attrs.get('units', '')
cbar.set_label(r"Biomass (mgC m$^{-2}$)", fontsize=12)

# Extract human-readable time
time_str = str(data_slice.time.values).split('T')[0]
plt.title(f"{label} | {spinup} | {time_str}", fontsize=12, pad=10)

# Save output
output_filename = f"{variable}_{spinup}_{date_str}_spatial.png"
plt.savefig(os.path.join(out_dir,output_filename), dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved static plot to {output_filename}")
ds.close()

In [ ]:
# Generate animation
# --- Configuration ---
base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
out_dir = "/export/lv9/projects/dws/results/animation/benthos"

#variable = "Y1c" 
#label = "Epibenthos (Y1)"

#variable = "Y2c" 
#label = "Deposit feeders (Y2)"

variable = "Y3c" 
label = "Suspension feeders (Y3)"

#variable = "Y5c" 
#label = "Endobenthos (Y5)"

frames_dir = os.path.join(out_dir, f"{variable}_frames")
os.makedirs(frames_dir, exist_ok=True)

# Fixed map window: Dutch Wadden Sea to Ems-Dollard
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# Fixed color scale configuration to prevent map flickering
VMIN = 10.0
VMAX = 100000.0

log_norm = mcolors.LogNorm(vmin=VMIN, vmax=VMAX)

#spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])
spinups = ["spinup_07", "spinup_08"]
frame_counter = 2190

for spinup in spinups:
    spinup_dir = os.path.join(base_dir, spinup)
    nc_files = sorted(glob.glob(os.path.join(spinup_dir, "dws_500m.3d.*.nc")))
    for file_path in nc_files:
        print(f"Processing frames from: {file_path}")

        with xr.open_dataset(file_path) as ds:
            # Fix curvilinear coordinate mesh dimensions
            lon_da = ds.lonc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')
            lat_da = ds.latc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')
            lon = lon_da.values
            lat = lat_da.values
            
            data_var = ds[variable]
            if 'level' in data_var.dims:
                data_var = data_var.isel(level=-1)
            #units = ds[variable].attrs.get('units', '')
            
            for t_idx in range(len(data_var.time)-1):
                data_slice = data_var.isel(time=t_idx)
                
                fig = plt.figure(figsize=(8, 6))
                ax = plt.axes(projection=ccrs.Mercator())
                ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())
                
                # Draw imagery tile set underneath
                ctx.add_basemap(ax, 
                                crs=ax.projection.to_string(), 
                                #source=ctx.providers.Esri.WorldImagery,
                                source=ctx.providers.Esri.WorldShadedRelief,
                                zorder=1)
                
                txt = ax.texts[-1]
                txt.set_position([0.99,0.02])
                txt.set_ha('right')
                
                # Plot curvilinear layer
                mesh = ax.pcolormesh(lon, lat, data_slice.values,
                                     transform=ccrs.PlateCarree(),
                                     cmap=cmocean.cm.matter,
                                     norm=log_norm,
                                     shading='gouraud',
                                     zorder=2)
                

                cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
                cbar.ax.tick_params(labelsize=10)
                cbar.set_label(r"Biomass (mgC m$^{-2}$)", fontsize=12)
                
                time_str = str(ds.time.values[t_idx])[:10]
                plt.title(f"{label} | {spinup} | {time_str}", fontsize=12, pad=10)
                
                frame_path = os.path.join(frames_dir, f"frame_{frame_counter:04d}.png")
                plt.savefig(frame_path, dpi=200, bbox_inches='tight')
                plt.close(fig)
                
                frame_counter += 1        

In [ ]:
#variable = "Y1c" 
#label = "Epibenthos (Y1)"

#variable = "Y2c" 
#label = "Deposit feeders (Y2)"

#variable = "Y3c" 
#label = "Suspension feeders (Y3)"

variable = "Y5c" 
label = "Endobenthos (Y5)"
out_dir = "/export/lv9/projects/dws/results/animation/benthos"
frames_dir = os.path.join(out_dir, f"{variable}_frames")

# --- Generate Animated GIF ---
print("Compiling frames into an animated GIF...")

output_gif = os.path.join(out_dir, f"{variable}_spinup.gif")

frame_paths = sorted(
    [
        os.path.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.startswith("frame_") and os.path.isfile(os.path.join(frames_dir, f))
    ]
)

if frame_paths:
    print(f"Compiling frames into {output_gif}...")

    frames = [iio.imread(f) for f in frame_paths]
    
    iio.imwrite(
        output_gif,
        frames,
        duration=25, 
        loop=0
    )
    
    print("Animation saved successfully.")